In [20]:
from google.colab import drive
drive.mount('/content/drive')


import pandas as pd
stock_ledger = pd.read_csv('/content/drive/MyDrive/Cadetx Project/HeavySuppliersWarehouseDatasets/stock_ledger.csv')
stock_ledger[['product_id','branch_id','movement_type', 'quantity']].isnull().sum()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,0
product_id,0
branch_id,0
movement_type,0
quantity,0


In [21]:
out_totals = stock_ledger[stock_ledger['movement_type'] == 'OUT'].groupby('product_id')['quantity'].sum()
in_totals = stock_ledger[stock_ledger['movement_type'] == 'IN'].groupby('product_id')['quantity'].sum()

products_never_sold = set(in_totals.index) - set(out_totals.index)
products_never_delivered = set(out_totals.index) - set(in_totals.index)

print("Products with IN but zero OUT:", products_never_sold)
print("Products with OUT but zero IN:", products_never_delivered)

Products with IN but zero OUT: set()
Products with OUT but zero IN: set()


In [22]:
stock_ledger['movement_date'] = pd.to_datetime(stock_ledger['movement_date'])
stock_ledger['month'] = stock_ledger['movement_date'].dt.to_period('M')

out_idx = set(stock_ledger[stock_ledger['movement_type']=='OUT'].groupby(['product_id','month']).size().index)
in_idx = set(stock_ledger[stock_ledger['movement_type']=='IN'].groupby(['product_id','month']).size().index)

only_in = in_idx - out_idx
only_out = out_idx - in_idx

print("Months with IN only (no OUT that month):", len(only_in))
print("Months with OUT only (no IN that month):", len(only_out))
print("Months with both:", len(in_idx & out_idx))

Months with IN only (no OUT that month): 0
Months with OUT only (no IN that month): 0
Months with both: 2190


In [23]:
stock_ledger['movement_type'].value_counts()

,count
movement_type,
IN,127611
OUT,107165
ADJUSTMENT,2454


In [24]:
adjustments = stock_ledger[stock_ledger['movement_type'] == 'ADJUSTMENT']
print(adjustments['quantity'].describe())
print(adjustments['reference_type'].value_counts())

count    2454.000000
mean       27.298289
std        13.247121
min         5.000000
25%        16.000000
50%        27.000000
75%        38.000000
max        50.000000
Name: quantity, dtype: float64
reference_type
ADJ    2454
Name: count, dtype: int64


In [25]:
clean = stock_ledger[stock_ledger['movement_type'].isin(['IN', 'OUT'])]

m1_in = clean[clean['movement_type'] == 'IN'].groupby('product_id')['quantity'].sum()
m1_out = clean[clean['movement_type'] == 'OUT'].groupby('product_id')['quantity'].sum()
ratio_m1 = (m1_in / m1_out).sort_index()

In [26]:
pivot = clean.pivot_table(index='product_id', columns='movement_type', values='quantity', aggfunc='sum')
ratio_m2 = (pivot['IN'] / pivot['OUT']).sort_index()

In [27]:
diff = (ratio_m1 - ratio_m2).abs()
print("Max difference between methods:", diff.max())
print("Any mismatches?", (diff > 0.0001).any())

Max difference between methods: 0.0
Any mismatches? False


In [28]:
inbound = stock_ledger[stock_ledger['movement_type'] == 'IN']

stats = inbound.groupby('product_id')['quantity'].agg(['mean', 'median', 'std', 'max', 'count'])
stats['max_to_mean_ratio'] = stats['max'] / stats['mean']
print(stats.sort_values('max_to_mean_ratio', ascending=False).head(10))

                  mean  median        std  max  count  max_to_mean_ratio
product_id                                                              
P027        156.429742   155.0  80.689783  300   4270           1.917794
P008        157.442221   157.0  80.752728  300   4197           1.905461
P026        158.057319   157.0  80.872760  300   4222           1.898046
P018        158.256635   158.0  81.680809  300   4333           1.895655
P013        158.671003   159.0  80.085239  300   4228           1.890705
P024        158.934638   158.0  80.612223  300   4345           1.887568
P014        159.033288   158.0  80.776877  300   4386           1.886398
P019        159.201547   159.5  81.654930  300   4138           1.884404
P016        159.267632   159.0  81.646668  300   4211           1.883622
P003        159.295360   161.0  80.935754  300   4310           1.883294


In [29]:
outbound = stock_ledger[stock_ledger['movement_type'] == 'OUT']
print(outbound['quantity'].describe())

count    107165.000000
mean         10.496263
std           5.776647
min           1.000000
25%           5.000000
50%          11.000000
75%          16.000000
max          20.000000
Name: quantity, dtype: float64


In [30]:
span = stock_ledger.groupby('product_id')['movement_date'].agg(['min', 'max'])
span['days'] = (pd.to_datetime(span['max']) - pd.to_datetime(span['min'])).dt.days
print(span['days'].describe())

count      30.000000
mean     2213.033333
std         3.537809
min      2205.000000
25%      2210.250000
50%      2213.500000
75%      2216.000000
max      2219.000000
Name: days, dtype: float64


In [31]:
for pid in ['P029', 'P003', 'P024']:
    print(f"--- {pid} ---")
    subset = stock_ledger[(stock_ledger['product_id'] == pid) & (stock_ledger['movement_type'].isin(['IN','OUT']))]
    print(subset.groupby('movement_type')['quantity'].agg(['sum', 'mean', 'count']))
    print()

--- P029 ---
                  sum        mean  count
movement_type                           
IN             673125  159.508294   4220
OUT             38648   10.691010   3615

--- P003 ---
                  sum        mean  count
movement_type                           
IN             686563  159.295360   4310
OUT             37920   10.509978   3608

--- P024 ---
                  sum        mean  count
movement_type                           
IN             690571  158.934638   4345
OUT             35924   10.397685   3455



In [32]:
!find /content/drive/MyDrive -iname "stock_ledger.csv"


/content/drive/MyDrive/Cadetx Project/HeavySuppliersWarehouseDatasets/stock_ledger.csv


In [33]:
stock_ledger = pd.read_csv('/content/drive/MyDrive/Cadetx Project/HeavySuppliersWarehouseDatasets/stock_ledger.csv')